In [ ]:
import os
import numpy as np
import h5py
import matplotlib.pyplot as plt
import scienceplots
import scipy
from dotenv import load_dotenv
from tqdm import tqdm

import tensorstore as ts

load_dotenv()
PATH = os.getenv("ROOT_PATH")

plt.style.use(['science', 'no-latex'])

def format_ax(ax):
  for spine in ax.spines.values():
    spine.set_linewidth(1.2)
  ax.spines['top'].set_visible(False)
  ax.spines['right'].set_visible(False)
  ax.spines['bottom'].set_visible(False)
  ax.tick_params(which='minor', length=0)
  ax.tick_params(axis='both', labelsize=12)
  for spine in ax.spines.values():
    spine.set_visible(False)
  ax.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
  ax.tick_params(axis='y', which='both', left=False, right=False, direction="out", width=1.2)

In [ ]:
subject_id = "06"

traces = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'file:///{PATH}/ts_files/subject_{subject_id}_traces.zarr'
    # 'kvstore': 'gs://zapbench-release/volumes/20240930/traces/'
}).result()

s = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'file:///{PATH}/ts_files/subject_{subject_id}_stimuli.zarr'
    # 'kvstore': 'gs://zapbench-release/volumes/20240930/traces/'
}).result()

coordinates = ts.open({
    'open': True,
    'driver': 'zarr3',
    'kvstore': f'file:///{PATH}/ts_files/subject_{subject_id}_coordinates.zarr'
    # 'kvstore': 'gs://zapbench-release/volumes/20240930/traces/'
}).result()

traces = traces.read().result()
s = s.read().result()
coordinates = coordinates.read().result()

f = h5py.File(f'/{PATH}/Additional_mat_files/MaskDatabase.mat', 'r')
names = f['MaskDatabaseNames']
names = [(i, "".join([chr(c[0]) for c in f[name[0]]])) for i, name in enumerate(names)]

In [ ]:
def linear_to_3d_matlab(linear_idx, width, height):
  idx = linear_idx - 1
  i = idx % width
  j = (idx // width) % height
  k = idx // (width * height)
  return i, j, k

def matlab_to_linear(i, j, k, width, height):
  linear_idx = i + j * width + k * width * height + 1
  return linear_idx


reference_anat = scipy.io.loadmat(f'/{PATH}/Additional_mat_files/ReferenceBrain.mat')['anat_stack_norm']

ir = f['MaskDatabase']['ir'][:]  # Row indices (linear voxel indices)
jc = f['MaskDatabase']['jc'][:]  # Column pointers
data = f['MaskDatabase']['data'][:]  # Should be all ones

mask = np.zeros_like(reference_anat)
mask_shape = mask.shape

# mask_idx_list = [77, 66, 67, 68, 69, 70, 71, 101, 102, 103, 106]
mask_idx_list = [106] # visual processing
# mask_idx_list = [96, 97, 200] # eye movement control
for mask_idx in mask_idx_list:
  start_idx = jc[mask_idx]
  end_idx = jc[mask_idx + 1]
  for i in ir[start_idx:end_idx]:
    i_x, i_y, i_z = linear_to_3d_matlab(i, mask_shape[0], mask_shape[1])
    mask[i_x, i_y, i_z] = 1


valid_coordinate_i = []
for i, c in enumerate(coordinates):
  if mask[c.astype(int)[0], c.astype(int)[1], c.astype(int)[2]]:
    valid_coordinate_i.append(i)
valid_coordinate_i = np.array(valid_coordinate_i)

In [ ]:
valid_coordinate_i.shape

Try sbi with neural trace data as-is

In [ ]:
import torch
from sbi.utils import BoxUniform
from sbi.inference import NPE

_ = torch.manual_seed(0)

In [ ]:
# traces_selected = traces[:, valid_coordinate_i]
traces_selected = traces

In [ ]:
valid_coordinate_i_h = np.mean(traces_selected, 0) > 0.15

valid_coordinate_i_h_l = (coordinates[valid_coordinate_i[valid_coordinate_i_h]][:, 1] < 300)
valid_coordinate_i_h_r = (coordinates[valid_coordinate_i[valid_coordinate_i_h]][:, 1] > 300)

valid_coordinate_i[valid_coordinate_i_h][valid_coordinate_i_h_l]

In [ ]:
context = 4
n_d = context
horizon = 16
n_neurons = traces_selected.shape[-1]
traces_selected.shape[0]-context-horizon
trace_data_x = np.stack([traces_selected[i:i+context] for i in range(0, traces_selected.shape[0]-context-horizon, 10)])
trace_data_theta = np.stack([traces_selected[i+context+horizon] for i in range(0, traces_selected.shape[0]-context-horizon, 10)])
trace_data_x = trace_data_x.transpose(0, 2, 1).reshape(trace_data_x.shape[0]*n_neurons, context)
trace_data_theta = trace_data_theta.ravel()

In [ ]:
trace_data_x_train = torch.tensor(trace_data_x[::300])
trace_data_theta_train = torch.tensor(trace_data_theta[::300, None])
trace_data_x_train.shape, trace_data_theta_train.shape

In [ ]:
prior = BoxUniform(low=-0.25 * torch.ones(n_d), high=3.0 * torch.ones(n_d))
inference = NPE(prior=prior)

In [ ]:
inference = inference.append_simulations(trace_data_theta_train, trace_data_x_train)
density_estimator = inference.train()

In [ ]:
posterior = inference.build_posterior()

Test on example traces

In [ ]:
selected_ix = [50765, 50772, 50787, 50797, 50802, 51057, 54002, 54044, 54360, 54420, 54466, 54485, 54592, 54622, 54751, 54762, 54765, 54787, 54793, 54795, 54812, 54818, 55189, 55221, 55313, 55374, 55430, 57621, 58133, 58299, 58305, 58365, 58495, 58513, 58540, 58568, 58587, 58621, 58691, 58752, 58758, 58787, 58801, 58815, 59076, 59105, 59162, 59183, 59198, 59203, 59226, 59252, 59256, 59286, 59287, 59337, 59743, 62071, 62128, 62179, 62212, 62304, 62330, 62331, 62332, 62337, 62539, 62545, 62584, 62634, 62980, 63061, 63105, 63106, 63140, 63162, 63197, 63215, 63259, 63282, 63374, 63502, 63507, 65802, 65882, 65984, 65999, 66017, 66101, 66125, 66131, 66149, 66228, 66229, 66252, 66267, 66271, 66385, 66398, 66431, 66593, 66976, 67137, 69409, 69415, 69478, 69549, 69646, 69691, 69702, 69819, 69839, 69898, 69978, 70103, 70126, 70143, 70151, 70366, 70428, 70523, 70591, 72675, 72719, 72913, 72989, 73041, 73045, 73051, 73063, 73075, 73140, 73259, 73261, 73292, 73402, 73406, 73543, 73640, 73770, 74311, 76433, 76515, 76543, 76640, 76648, 76692, 76742, 76900, 77719, 79635, 79722, 79810, 79875, 80012, 80077, 80209, 80285, 83107, 83199, 85955, 86031, 86069, 86128]

In [ ]:
n_ix = 50
fig, axs = plt.subplots(n_ix, 1, figsize=(10, n_ix//2), dpi=500, squeeze=False)
axs = axs.flatten()

for i, neuron_ix in enumerate(tqdm(selected_ix[:n_ix])):
    predicted_trace_i = []
    theta_hat_i = []
    for j in range(0, 500):
        x_obs = torch.tensor(traces[:, neuron_ix][j:j+context])
        theta_hat = posterior.sample((100,), x=x_obs, show_progress_bars=False)
        theta_hat_i.append(theta_hat.numpy())
        predicted_trace_i.append(theta_hat.mean().item())
    theta_hat_i = np.array(theta_hat_i).squeeze(-1)

    ax = axs[i]
    ax.plot(traces[:500+context, neuron_ix], 'k')
    ax.plot(np.arange(context, theta_hat_i.shape[0]+context), np.quantile(theta_hat_i, 0.5, axis=-1), 'r')
    ax.fill_between(
        np.arange(context, theta_hat_i.shape[0]+context),
        np.quantile(theta_hat_i, 0.05, axis=-1),
        np.quantile(theta_hat_i, 0.95, axis=-1),
        color='red',
        alpha=0.1,
        linewidth=0,
    )
    format_ax(ax)

NameError: name 'plt' is not defined

Next, we train exactly the same network, but condition on visual input stimulus

In [ ]:
stimuli_x = np.where(s==1)[-1].astype(np.float32)

In [ ]:
trace_data_x = np.stack([traces[i:i+context] for i in range(0, traces.shape[0]-context-horizon, 100)])
stimuli_x_data = np.stack([stimuli_x[i:i+context] for i in range(0, stimuli_x.shape[0]-context-horizon, 100)])
trace_data_theta = np.stack([traces[i+context+horizon] for i in range(0, traces.shape[0]-context-horizon, 100)])

trace_data_x = np.concatenate([trace_data_x, np.broadcast_to(stimuli_x_data[..., None], trace_data_x.shape)], axis=1)
trace_data_x = trace_data_x.transpose(0, 2, 1).reshape(trace_data_x.shape[0]*n_neurons, context*2)
trace_data_theta = trace_data_theta.ravel()

In [ ]:
trace_data_x_train = torch.tensor(trace_data_x[::500])
trace_data_theta_train = torch.tensor(trace_data_theta[::500, None])
trace_data_x_train.shape, trace_data_theta_train.shape

In [ ]:
prior = BoxUniform(low=-0.25 * torch.ones(n_d), high=3.0 * torch.ones(n_d))
inference = NPE(prior=prior)

In [ ]:
inference = inference.append_simulations(trace_data_theta_train, trace_data_x_train)
density_estimator = inference.train()

In [ ]:
posterior = inference.build_posterior()

Test on example traces

In [ ]:
n_ix = 50
fig, axs = plt.subplots(n_ix, 1, figsize=(10, n_ix//2), dpi=500, squeeze=False)
axs = axs.flatten()

for i, neuron_ix in enumerate(tqdm(selected_ix[:n_ix])):
    predicted_trace_i = []
    theta_hat_i = []
    for j in range(0, 500):
        x_obs = torch.tensor(np.concatenate([traces[:, neuron_ix][j:j+context], stimuli_x[j:j+context]]))
        theta_hat = posterior.sample((100,), x=x_obs, show_progress_bars=False)
        theta_hat_i.append(theta_hat.numpy())
        predicted_trace_i.append(theta_hat.mean().item())
    theta_hat_i = np.array(theta_hat_i).squeeze(-1)

    ax = axs[i]
    ax.plot(traces[:500+context, neuron_ix], 'k')
    ax.plot(stimuli_x[:500+context]/10, 'm')
    ax.plot(np.arange(context, theta_hat_i.shape[0]+context), np.quantile(theta_hat_i, 0.5, axis=-1), 'r')
    ax.fill_between(
        np.arange(context, theta_hat_i.shape[0]+context),
        np.quantile(theta_hat_i, 0.05, axis=-1),
        np.quantile(theta_hat_i, 0.95, axis=-1),
        color='red',
        alpha=0.1,
        linewidth=0,
    )
    format_ax(ax)

plt.tight_layout()

This did not seem to be consistently useful to change the overall stats. Now we need to find a clever way to condition on the "neuron type". i'm thinking of region (tectal neuropil) and maybe cluster the neurons by activation pattern first (this will hopefully happen automatically later, but now we just want an informed prior)

First we check whether the active neurons tend to be in phase or out of phase with the input in the tectum neuropil, and if there's a systematic difference between the hemispheres – doesn't look like it, at first glance

In [ ]:
selected_valid_coordinates_i = valid_coordinate_i[valid_coordinate_i_h][valid_coordinate_i_h_r]
n_ix = len(selected_valid_coordinates_i)
n_ix = 20
fig, axs = plt.subplots(n_ix, 1, figsize=(10, n_ix//2), dpi=500, squeeze=False)
axs = axs.flatten()

for i, neuron_ix in enumerate(tqdm(selected_valid_coordinates_i[:n_ix])):
    ax = axs[i]
    ax.plot(traces[:500+context, neuron_ix], 'k')
    ax.plot(stimuli_x[:500+context]/10, 'm')
    format_ax(ax)

Do these neurons (active, tectum neuropil, left hemisphere) cluster based on their response to the stimulus and their location within the tectal neuropil (3d)?

In [ ]:
len(selected_valid_coordinates_i)

In [ ]:
selected_valid_coordinates_i = valid_coordinate_i[valid_coordinate_i_h][valid_coordinate_i_h_r]
n_ix = len(selected_valid_coordinates_i)
n_ix = 26
fig, axs = plt.subplots(n_ix, 1, figsize=(10, n_ix//2), dpi=500, squeeze=False)
axs = axs.flatten()

for i, neuron_ix in enumerate(tqdm(selected_valid_coordinates_i[:n_ix])):
    ax = axs[i]
    ax.plot(traces[:, neuron_ix][np.where(stimuli_x == 2)[0]], 'k')
    format_ax(ax)

There's no obvious pattern; now let's check whether neurons that tend to be more active on average during stimulus 1 vs 2 tend to cluster with coordinate

In [ ]:
type = []
for i, neuron_ix in enumerate(tqdm(selected_valid_coordinates_i[:n_ix])):
  type.append(np.mean(traces[:, neuron_ix][stimuli_x == 3]) > np.mean(traces[:, neuron_ix][stimuli_x == 2]))
  # type[-1] = type[-1]*np.mean(traces[:, neuron_ix][stimuli_x == 1])

type_2 = []
for i, neuron_ix in enumerate(tqdm(selected_valid_coordinates_i[:n_ix])):
  type_2.append(np.mean(traces[:, neuron_ix][stimuli_x == 2]) > np.mean(traces[:, neuron_ix][stimuli_x == 1]))
  # type_2[-1] = type_2[-1]*np.mean(traces[:, neuron_ix][stimuli_x == 2])

In [ ]:
import pyvista as pv
import numpy as np

# Extract the relevant coordinates
coords = coordinates[selected_valid_coordinates_i[:n_ix]]

# Convert types to numpy arrays for coloring
type_arr = np.array(type)
type_2_arr = np.array(type_2)

# First 3D scatter: colored by 'type'
plotter = pv.Plotter()
plotter.add_points(coords, scalars=type_arr.astype(float), render_points_as_spheres=True, point_size=10, cmap="coolwarm")
plotter.add_axes()
plotter.show(title="Coordinates colored by type (stim 1 > stim 2)")

# Second 3D scatter: colored by 'type_2'
plotter = pv.Plotter()
plotter.add_points(coords, scalars=type_2_arr.astype(float), render_points_as_spheres=True, point_size=10, cmap="coolwarm")
plotter.add_axes()
plotter.show(title="Coordinates colored by type_2 (stim 2 > stim 1)")